### 

In [ ]:
train_file = "/root/autodl-tmp/movielens/train.txt"
user2index_file = "/root/autodl-tmp/movielens/user2index.txt"
item2index_file = "/root/autodl-tmp/movielens/item2index.txt"
movies_file = "/root/autodl-tmp/movielens/movies.dat"


User = {}
user2index = {}
item2index = {}
movies = {}


with open(item2index_file, 'r') as f:
    next(f) 
    for line in f:
        mapped_item_id, item_id = line.strip().split('\t')
        item2index[mapped_item_id] = item_id


with open(movies_file, 'r', encoding='latin1') as f:
    for line in f:
        item_id, title, _ = line.strip().split("::", 2)
        movies[item_id] = title


with open(train_file, 'r') as f:
    next(f) 
    for line in f:
        user_id, mapped_item_id, interaction = line.strip().split('\t')
        if interaction == '1':  
            item_id = item2index.get(mapped_item_id)
            if user_id and item_id:  
                item_title = movies.get(item_id, "Unknown Title")
                if user_id not in User:
                    User[user_id] = []
                User[user_id].append({'item_id':item_id, 
                                      'item_title':item_title})

In [2]:
User["0"]

[{'item_id': '3105', 'item_title': 'Awakenings (1990)'},
 {'item_id': '1029', 'item_title': 'Dumbo (1941)'},
 {'item_id': '1907', 'item_title': 'Mulan (1998)'},
 {'item_id': '1207', 'item_title': 'To Kill a Mockingbird (1962)'},
 {'item_id': '2018', 'item_title': 'Bambi (1942)'},
 {'item_id': '527', 'item_title': "Schindler's List (1993)"},
 {'item_id': '2028', 'item_title': 'Saving Private Ryan (1998)'},
 {'item_id': '2797', 'item_title': 'Big (1988)'},
 {'item_id': '594', 'item_title': 'Snow White and the Seven Dwarfs (1937)'},
 {'item_id': '608', 'item_title': 'Fargo (1996)'},
 {'item_id': '1097', 'item_title': 'E.T. the Extra-Terrestrial (1982)'},
 {'item_id': '48', 'item_title': 'Pocahontas (1995)'},
 {'item_id': '1270', 'item_title': 'Back to the Future (1985)'},
 {'item_id': '938', 'item_title': 'Gigi (1958)'},
 {'item_id': '2791', 'item_title': 'Airplane! (1980)'},
 {'item_id': '1836', 'item_title': 'Last Days of Disco, The (1998)'},
 {'item_id': '3186', 'item_title': 'Girl, In

In [ ]:
import json
def create_interaction_prompt(User, filename):
    prompts = []
    for userid in User:
        # Sort by unixReviewTime from high to low
        #User[userid] = sorted(User[userid], key=lambda x: x['time'], reverse=True)

        total_words = 0
        reviews = []
        for entry in User[userid]:
            item_title = entry["item_title"]
            words = item_title.split()

            truncated_review = item_title
            review_length = len(truncated_review.split())

            if review_length + total_words > 2000:
                break

            reviews.append(f'{{"title": "{item_title}"}}')
            total_words = total_words + review_length

        prompt = f'userid: {userid}\nInteracted items: \n[\n' + ',\n'.join(reviews) + '\n]'
        prompts.append({"userid": int(userid),"prompt": prompt})
        
    prompts = sorted(prompts, key=lambda x: x["userid"])
    # Save the prompts to the specified filename
    with open(filename, 'w') as f:
        json.dump(prompts, f, indent=4)

In [23]:
create_interaction_prompt(User, "/root/autodl-tmp/movielens/crossaug_movielens_train_.json")

### 

In [ ]:
import gzip
import json
import pandas as pd
from tqdm import tqdm


def load_item2index(file_path):
    item_dict = {}
    with open(file_path, 'r') as fin:
        lines = fin.readlines()
        for line in lines[1:]:
            item_index, item_id = line.strip().split()
            item_dict[item_id] = int(item_index)
    return item_dict


def parse_meta(path, item_dict):
    Item = defaultdict(list)
    
    with gzip.open(path, 'rt', encoding='utf-8') as fin:
        for line in tqdm(fin, desc="Processing meta data"):
            meta_data = json.loads(line)
            asin = meta_data['asin']
            title = meta_data.get('title', ' ')
            description = meta_data.get('description', ' ')
            
   
            item_id = item_dict.get(asin)  
            
            if item_id is not None: 
                Item[item_id].append({
                    'title': title,
                    'description': description,
                    'item_id': asin
                })
    
    return Item


file_path_meta = '/root/autodl-tmp/arts/meta_Arts_Crafts_and_Sewing.json.gz'
file_path_item2index = '/root/autodl-tmp/arts/item2index.txt'

item_dict = load_item2index(file_path_item2index) 
Item = parse_meta(file_path_meta, item_dict)

print("Meta processing complete.")


Processing meta data: 302988it [00:18, 16399.21it/s]

Meta processing complete.


In [18]:
Item[0]

[{'title': 'HP C6049A Iron-On Transfers, 8-1/2 x 11, White (Pack of 12)',
  'description': ['HP Iron-on Transfers are suitable for cotton or cotton/poly blend fabrics. These iron-on, cool-peel inkjet transfers are specially designed to produce vivid color images and graphics on white or light-colored fabrics. Whats in the box: 12 sheets of 8.5 x 11 inch paper for transferring to fabric.',
   '',
   "HP's iron-on, cool-peel inkjet transfers make it easy and fun to design t-shirts, mouse pads, caps and other customized items for your business, family, sports team or organization. These transfers work best on white fabrics and work fine on light-colored fabric.",
   'Whether for professional, creative, or everyday use, HP helps take the guesswork out of finding the perfect inkjet or LaserJet paper for any project.',
   'Not all papers are created equal. Designed with each element of the HP printing process in mind, HP papers, toners, and inks are specially engineered to work together to d

### 

In [ ]:

def load_train_data(file_path):
    train_data = defaultdict(set)
    with open(file_path, 'r') as fin:
        lines = fin.readlines()
        for line in lines[1:]: 
            user_index, item_index, _ = line.strip().split()
            train_data[int(user_index)].add(int(item_index))  
    return train_data

def filter_user_data(User, train_data, item_dict):
    for user_id in list(User.keys()):  
        valid_item_ids = train_data.get(user_id, set())  
        User[user_id] = [review for review in User[user_id] if item_dict.get(review['itemid']) in valid_item_ids]
        
       
        if not User[user_id]:
            del User[user_id]
    return User


In [20]:
file_path_train = "/root/autodl-tmp/arts/train.txt"
train_data = load_train_data(file_path_train)
User = filter_user_data(User, train_data, item_dict)

In [21]:
User[0]

[{'time': 1475020800,
  'itemid': 'B00115PN6E',
  'reviewText': 'nice colors and lots of them',
  'summary': 'Four Stars',
  'rating': 4.0,
  'reviewID': 'A100UXMXYOQU1X'},
 {'time': 1475020800,
  'itemid': 'B00115PN6E',
  'reviewText': 'nice colors and lots of them',
  'summary': 'Four Stars',
  'rating': 4.0,
  'reviewID': 'A100UXMXYOQU1X'},
 {'time': 1475020800,
  'itemid': 'B00AFRWILM',
  'reviewText': 'work great with large earrings!',
  'summary': 'Five Stars',
  'rating': 5.0,
  'reviewID': 'A100UXMXYOQU1X'},
 {'time': 1477526400,
  'itemid': 'B00OSP7J3O',
  'reviewText': 'These help so much.  Really like them.',
  'summary': 'Really like them.',
  'rating': 5.0,
  'reviewID': 'A100UXMXYOQU1X'},
 {'time': 1525132800,
  'itemid': 'B000WWIPEE',
  'reviewText': "Works great!  You can't put 20 pieces of paper in it but I only use it occasionally so it's perfect for me.",
  'summary': 'Perfect for me!',
  'rating': 5.0,
  'reviewID': 'A100UXMXYOQU1X'},
 {'time': 1532217600,
  'itemid

In [22]:
len(User)

9404

In [ ]:
def add_title_to_reviews(User, Item):
    empty_title_count = 0
    for user_id, reviews in User.items():
        for review in reviews:
            asin = review['itemid']
            
            item_index = item_dict.get(asin)
            
            item_info = Item.get(asin)
            if item_index is not None:
                item_info = Item.get(item_index)
                if item_info:
                    title = item_info[0].get('title', ' ').strip()
                    description = item_info[0].get('description', ' ')
                    if isinstance(description, list): 
                        description = " ".join(description).strip()
                    else: 
                        description = description.strip()
                        
                    if any(len(word) > 50 for word in title.split()):
                        title = description
                        
                    
                    if title:
                        review['title'] = title
                    elif description:
                        if any(len(word) > 50 for word in description.split()):
                            description = 'none'
                            empty_title_count += 1
                        review['title'] = description
                    else:
                        review['title'] = 'none'
                        empty_title_count += 1
                else:
                    review['title'] = 'none'
                    empty_title_count += 1
    
    return empty_title_count

In [ ]:

empty_title_count = add_title_to_reviews(User, Item)

In [25]:
User[0]

[{'time': 1475020800,
  'itemid': 'B00115PN6E',
  'reviewText': 'nice colors and lots of them',
  'summary': 'Four Stars',
  'rating': 4.0,
  'reviewID': 'A100UXMXYOQU1X',
  'title': 'Prism 6-Strand Floss Jumbo Pack 8.7yd 105/Pkg, Assorted Colors'},
 {'time': 1475020800,
  'itemid': 'B00115PN6E',
  'reviewText': 'nice colors and lots of them',
  'summary': 'Four Stars',
  'rating': 4.0,
  'reviewID': 'A100UXMXYOQU1X',
  'title': 'Prism 6-Strand Floss Jumbo Pack 8.7yd 105/Pkg, Assorted Colors'},
 {'time': 1475020800,
  'itemid': 'B00AFRWILM',
  'reviewText': 'work great with large earrings!',
  'summary': 'Five Stars',
  'rating': 5.0,
  'reviewID': 'A100UXMXYOQU1X',
  'title': '50 Piece HypoAllergenic Bullet Clutch Earring Backs with Pad, SilverTone'},
 {'time': 1477526400,
  'itemid': 'B00OSP7J3O',
  'reviewText': 'These help so much.  Really like them.',
  'summary': 'Really like them.',
  'rating': 5.0,
  'reviewID': 'A100UXMXYOQU1X',
  'title': 'Dritz LoRan Stay-On Needle Puller, S

In [26]:
empty_title_count

0

### 

In [ ]:
def create_interaction_prompt(User, filename):
    prompts = []
    for userid in User:
        # Sort by unixReviewTime from high to low
        User[userid] = sorted(User[userid], key=lambda x: x['time'], reverse=True)

        total_words = 0
        reviews = []
        for entry in User[userid]:
            review_text = entry["reviewText"]
            words = review_text.split()
            if len(words) > 200:
                truncated_review = ' '.join(words[:200])
                summary = entry.get("summary", "")
                if summary:
                    truncated_review += ". The summary:" + summary
            else:
                truncated_review = review_text
                summary = entry.get("summary", "")
                if summary:
                    truncated_review += ". The summary:" + summary

            review_length = len(truncated_review.split())
            title = entry.get("title", "")

            # Handle the case where title is a list
            if isinstance(title, list):
                title = " ".join(title)

            if title:
                title_len = len(title.split())
            else:
                title_len = 0

            if title_len + total_words + review_length > 2500:
                break

            reviews.append(f'{{"title": "{title}", "review": "{truncated_review}"}}')
            total_words = total_words + review_length + title_len

        prompt = f'userid: {userid}\nInteracted items: \n[\n' + ',\n'.join(reviews) + '\n]'
        prompts.append({"userid": int(userid),"prompt": prompt})
        
    
    prompts = sorted(prompts, key=lambda x: x["userid"])
    # Save the prompts to the specified filename
    with open(filename, 'w') as f:
        json.dump(prompts, f, indent=4)

In [28]:
create_interaction_prompt(User, "/root/autodl-tmp/arts/crossaug_arts_train_.json")